In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [2]:
# Datasets and DataLoaders

from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = CIFAR10(root = "./data", train = True, download = True , transform = transform)
testset = CIFAR10(root = "./data", train = False, download = True , transform = transform)

100%|██████████| 170M/170M [05:15<00:00, 540kB/s]  


In [3]:
trainloader = DataLoader(trainset, batch_size = 64, shuffle = True)
testloader = DataLoader(testset, batch_size = 64)

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3,32, kernel_size = 3, padding =1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32,64, kernel_size = 3 , padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),
            nn.Linear(256,10)
        )
    
    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1)
        x = self.fc_layers(x)
        
        return x
        

In [5]:
model = CNN()

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [8]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images,labels in trainloader:
        optimizer.zero_grad()
        output = model.forward(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    print(f"epoch = {epoch+1}/{epoch} & loss = {epoch_training_loss/len(trainloader)}")

epoch = 1/0 & loss = 1.395101244187416
epoch = 2/1 & loss = 0.9462305145419162
epoch = 3/2 & loss = 0.763522336245193
epoch = 4/3 & loss = 0.6333205337109773
epoch = 5/4 & loss = 0.5267255968602417
epoch = 6/5 & loss = 0.4337761233301114
epoch = 7/6 & loss = 0.3509148477822009
epoch = 8/7 & loss = 0.2831699879711394
epoch = 9/8 & loss = 0.22539531301868998
epoch = 10/9 & loss = 0.1835778546483849


In [9]:
# Evaluate our CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _,predicted = torch.max(outputs,1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

    print(f"accuracy = {correct_labels / total_labels *100}")

accuracy = 72.45
